In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pandas as pd

base_path = "/content/drive/MyDrive/archive"

orders_path = f"{base_path}/orders.csv"
prior_path = f"{base_path}/order_products__prior.csv"
train_path = f"{base_path}/order_products__train.csv"
products_path = f"{base_path}/products.csv"

In [13]:
orders = pd.read_csv(orders_path)
prior = pd.read_csv(prior_path)
train = pd.read_csv(train_path)
products = pd.read_csv(products_path)

print("orders:", orders.shape)
print("prior:", prior.shape)
print("train:", train.shape)
print("products:", products.shape)

orders: (3421083, 7)
prior: (32434489, 4)
train: (1384617, 4)
products: (49688, 4)


In [14]:
top_1500_counts = prior["product_id"].value_counts().head(1500).reset_index()
top_1500_counts.columns = ["product_id", "count"]

top_product_ids = top_1500_counts["product_id"]

top_1500_counts.to_csv(f"{base_path}/top_1500_product_ids.csv", index=False)

print("top_1500_product_ids.csv 저장 완료")
print(top_1500_counts.head())

top_1500_product_ids.csv 저장 완료
   product_id   count
0       24852  472565
1       13176  379450
2       21137  264683
3       21903  241921
4       47209  213584


In [15]:
prior_top1500 = prior[prior["product_id"].isin(top_product_ids)].copy()
products_top1500 = products[products["product_id"].isin(top_product_ids)].copy()

prior_top1500.to_csv(f"{base_path}/order_products__prior_top1500.csv", index=False)
products_top1500.to_csv(f"{base_path}/products_top1500.csv", index=False)

print("order_products__prior_top1500.csv 저장 완료:", prior_top1500.shape)
print("products_top1500.csv 저장 완료:", products_top1500.shape)

order_products__prior_top1500.csv 저장 완료: (19776345, 4)
products_top1500.csv 저장 완료: (1500, 4)


In [16]:
valid_order_ids_top1500 = prior_top1500["order_id"].unique()

orders_top1500 = orders[orders["order_id"].isin(valid_order_ids_top1500)].copy()
orders_top1500.to_csv(f"{base_path}/orders_top1500.csv", index=False)

print("orders_top1500.csv 저장 완료:", orders_top1500.shape)

orders_top1500.csv 저장 완료: (3034651, 7)


In [17]:
train_top1500 = train[train["product_id"].isin(top_product_ids)].copy()
train_top1500.to_csv(f"{base_path}/order_products__train_top1500.csv", index=False)

print("order_products__train_top1500.csv 저장 완료:", train_top1500.shape)

order_products__train_top1500.csv 저장 완료: (816264, 4)


In [18]:
user_order_counts = orders_top1500.groupby("user_id")["order_id"].count()

valid_users = user_order_counts[user_order_counts >= 5].index

orders_top1500_user5 = orders_top1500[orders_top1500["user_id"].isin(valid_users)].copy()
orders_top1500_user5.to_csv(f"{base_path}/orders_top1500_user5.csv", index=False)

print("orders_top1500_user5.csv 저장 완료:", orders_top1500_user5.shape)
print("유효 사용자 수:", orders_top1500_user5['user_id'].nunique())

orders_top1500_user5.csv 저장 완료: (2873678, 7)
유효 사용자 수: 155461


In [19]:
valid_order_ids_user5 = orders_top1500_user5["order_id"].unique()

prior_top1500_user5 = prior_top1500[prior_top1500["order_id"].isin(valid_order_ids_user5)].copy()
prior_top1500_user5.to_csv(f"{base_path}/order_products__prior_top1500_user5.csv", index=False)

print("order_products__prior_top1500_user5.csv 저장 완료:", prior_top1500_user5.shape)

order_products__prior_top1500_user5.csv 저장 완료: (18898581, 4)


In [20]:
prior_with_orders = prior_top1500_user5.merge(
    orders_top1500_user5[["order_id", "user_id", "order_number"]],
    on="order_id",
    how="inner"
)

prior_with_orders_sorted = prior_with_orders.sort_values(
    ["user_id", "order_number", "add_to_cart_order"]
).copy()

prior_with_orders_sorted.to_csv(
    f"{base_path}/prior_with_orders_top1500_user5_sorted.csv",
    index=False
)

print("prior_with_orders_top1500_user5_sorted.csv 저장 완료:", prior_with_orders_sorted.shape)

prior_with_orders_top1500_user5_sorted.csv 저장 완료: (18898581, 6)


In [21]:
user_sequences = prior_with_orders_sorted.groupby("user_id")["product_id"].apply(list)

# 시퀀스 길이 6 이상만 남기고 싶으면 사용
# user_sequences = user_sequences[user_sequences.apply(len) >= 6]

user_sequences_df = user_sequences.reset_index()
user_sequences_df.columns = ["user_id", "product_sequence"]

user_sequences_df["product_sequence"] = user_sequences_df["product_sequence"].apply(
    lambda x: " ".join(map(str, x))
)

user_sequences_df.to_csv(f"{base_path}/user_sequences_top1500_user5.csv", index=False)

print("user_sequences_top1500_user5.csv 저장 완료:", user_sequences_df.shape)
print(user_sequences_df.head())

user_sequences_top1500_user5.csv 저장 완료: (155461, 2)
   user_id                                   product_sequence
0        1  196 14084 12427 196 12427 13176 196 12427 2513...
1        2  47766 20574 22474 16589 27344 30489 27966 1317...
2        3  9387 16797 39190 47766 21903 39922 24810 38596...
3        7  39275 45066 13249 31683 22963 14332 4920 22963...
4       10  46979 24852 27104 16797 31717 46979 20995 4301...


In [22]:
files_to_check = [
    "top_1500_product_ids.csv",
    "order_products__prior_top1500.csv",
    "products_top1500.csv",
    "orders_top1500.csv",
    "order_products__train_top1500.csv",
    "orders_top1500_user5.csv",
    "order_products__prior_top1500_user5.csv",
    "prior_with_orders_top1500_user5_sorted.csv",
    "user_sequences_top1500_user5.csv",
]

import os

for fname in files_to_check:
    path = f"{base_path}/{fname}"
    print(fname, "존재:", os.path.exists(path), "크기(bytes):", os.path.getsize(path) if os.path.exists(path) else "없음")

top_1500_product_ids.csv 존재: True 크기(bytes): 16713
order_products__prior_top1500.csv 존재: True 크기(bytes): 352173989
products_top1500.csv 존재: True 크기(bytes): 56620
orders_top1500.csv 존재: True 크기(bytes): 96098559
order_products__train_top1500.csv 존재: True 크기(bytes): 14550388
orders_top1500_user5.csv 존재: True 크기(bytes): 91156189
order_products__prior_top1500_user5.csv 존재: True 크기(bytes): 336563071
prior_with_orders_top1500_user5_sorted.csv 존재: True 크기(bytes): 507493332
user_sequences_top1500_user5.csv 존재: True 크기(bytes): 111015950


In [3]:
rec_df = pd.read_csv("/content/drive/MyDrive/archive/recommendations_top5_lstm.csv")

users_df = rec_df[["user_id"]].drop_duplicates().copy()
users_df["display_name"] = users_df["user_id"].apply(lambda x: f"Demo User {x}")

users_df.to_csv("/content/drive/MyDrive/archive/demo_users.csv", index=False)

In [4]:
orders_path = f"{base_path}/orders_top1500_user5.csv"
order_items_path = f"{base_path}/order_products__prior_top1500_user5.csv"

orders_df = pd.read_csv(orders_path)
order_items_df = pd.read_csv(order_items_path)

# 사용자별 최근 5개 주문만 남기기
recent_n = 5

orders_recent = (
    orders_df.sort_values(["user_id", "order_number"], ascending=[True, False])
    .groupby("user_id")
    .head(recent_n)
    .copy()
)

# 해당 주문에 속한 order_items만 남기기
valid_order_ids = orders_recent["order_id"].unique()
order_items_recent = order_items_df[order_items_df["order_id"].isin(valid_order_ids)].copy()

print("orders_recent:", orders_recent.shape)
print("order_items_recent:", order_items_recent.shape)

orders_recent: (777305, 7)
order_items_recent: (5044131, 4)


In [5]:
orders_recent.to_csv(f"{base_path}/demo_orders_recent5.csv", index=False)
order_items_recent.to_csv(f"{base_path}/demo_order_items_recent5.csv", index=False)